# Masked Autoencoders Are Effective Tokenizers for Diffusion Models

In [6]:
import torch
from PIL import Image
import numpy as np
import requests
import sys
from ..modelling.tokenizer import AEModel
from matplotlib import pyplot as plt

ImportError: attempted relative import with no known parent package

In [ ]:
def show_image(image, title=''):
    # image is [H, W, 3]
    assert image.shape[2] == 3
    plt.imshow(torch.clip((image * 0.5 + 0.5) * 255, 0, 255).int())
    plt.title(title, fontsize=16)
    plt.axis('off')
    return

## Load Model

In [ ]:
ae = AEModel.from_pretrained("MAETok/maetok-b-128")
# ae = AEModel.from_pretrained('/data2/lqh/workspace_pycharm/GenerateModel/continuous_tokenizer/SoftVQVAE/softvq-b-64-lasot')

ae = ae.eval()
if torch.cuda.is_available():
    ae = ae.cuda()
    device = torch.device('cuda')
    device_type = 'cuda'
else:
    device = torch.device('cpu')
    device_type = 'cpu'

## Load 256x256 Image

In [ ]:
# load an image
img_url = 'https://user-images.githubusercontent.com/11435359/147738734-196fd92f-9260-48d5-ba7e-bf103d29364d.jpg' # fox, from ILSVRC2012_val_00046145
# img_url = 'https://user-images.githubusercontent.com/11435359/147743081-0428eecf-89e5-4e07-8da5-a30fd73cc0ba.jpg' # cucumber, from ILSVRC2012_val_00047851
# img = Image.open(requests.get(img_url, stream=True).raw)
img = Image.open('/data2/lqh/data/lasot_vqvae/val_input/airplane-1/1020.jpg')
img = img.resize((256, 256))
img = np.array(img) / 255.

assert img.shape == (256, 256, 3)

# normalize by ImageNet mean and std
img = img - 0.5
img = img / 0.5

show_image(torch.tensor(img))

## 256x256 Reconstruction

In [ ]:
input_img = torch.tensor(img).permute(2, 0, 1).unsqueeze(0).float().to(device)
with torch.amp.autocast(device_type=device_type) and torch.no_grad():
    recon_img, _, _ = ae(input_img)
recon_img = recon_img[0].cpu().permute(1, 2, 0).numpy()

In [ ]:
# visualization
show_image(torch.tensor(recon_img))

# Load 512x512 Model and Image

In [ ]:
# ae = AEModel.from_pretrained("MAETok/maetok-b-128-512")
# ae = ae.eval()
# if torch.cuda.is_available():
#     ae = ae.cuda()
#     device = torch.device('cuda')
#     device_type = 'cuda'
# else:
#     device = torch.device('cpu')
#     device_type = 'cpu'

In [ ]:
# # load an image
# img_url = 'https://user-images.githubusercontent.com/11435359/147738734-196fd92f-9260-48d5-ba7e-bf103d29364d.jpg' # fox, from ILSVRC2012_val_00046145
# # img_url = 'https://user-images.githubusercontent.com/11435359/147743081-0428eecf-89e5-4e07-8da5-a30fd73cc0ba.jpg' # cucumber, from ILSVRC2012_val_00047851
# img = Image.open(requests.get(img_url, stream=True).raw)
# img = img.resize((512, 512))
# img = np.array(img) / 255.
#
# assert img.shape == (512, 512, 3)
#
# # normalize by ImageNet mean and std
# img = img - 0.5
# img = img / 0.5
#
# show_image(torch.tensor(img))